# 05 · Cost & Latency

> **Source notes:** `CostAndLatency.md`

This notebook makes the cost & latency formulas concrete:
- **Token counting & cost estimation** — tiktoken + pricing calculator
- **Context composition** — where tokens come from in a RAG pipeline
- **Conversation history leak** — quantifying the biggest cost driver
- **Prefix caching savings** — realistic calculation
- **Latency decomposition** — TTFT, generation, post-processing budgets

No LLM API key needed — all calculations use local token counting.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'tiktoken', '-q'], check=True)
print('Packages ready.')
import tiktoken
enc = tiktoken.get_encoding('cl100k_base')
def count_tokens(text): return len(enc.encode(text))
print(f"Test: 'hello world' = {count_tokens('hello world')} tokens")

## 1 · Cost Formula

```
Cost = (input_tokens x input_$/1M) + (output_tokens x output_$/1M)
```
Every architectural decision maps back to this formula.

In [ ]:
MODEL_PRICING = {
    'gpt-4o':            {'input': 5.00,  'output': 15.00, 'tier': 'frontier'},
    'gpt-4o-mini':       {'input': 0.15,  'output': 0.60,  'tier': 'mid'},
    'claude-3.5-sonnet': {'input': 3.00,  'output': 15.00, 'tier': 'frontier'},
    'claude-haiku':      {'input': 0.25,  'output': 1.25,  'tier': 'mid'},
    'llama-3-8b':        {'input': 0.05,  'output': 0.08,  'tier': 'open-source'},
}
def cost_per_call(inp, outp, model):
    p = MODEL_PRICING[model]
    return (inp * p['input'] + outp * p['output']) / 1_000_000

system_prompt    = 'You are a helpful assistant for Mamma Rosa Pizzeria. Answer only using the provided context.'
few_shot         = 'Q: Margherita allergens? A: Dairy and gluten.'
retrieved_chunks = ('Menu: Margherita 9.99 (gf +1.50), Pepperoni 11.99, Veggie 10.49. '
                    'Allergens: Pepperoni dairy/gluten/pork. Delivery: central all week, north Mon-Fri. Min order 15. '
                    'Hours: Mon-Thu 11am-10pm, Fri-Sat 11am-11pm, Sun noon-9pm.') * 4
user_message     = 'What is the cheapest gluten-free pizza for Monday delivery?'
typical_output   = 'Margherita with GF base is cheapest at 11.49. North zone delivery available Mondays.'

INPUT_PARTS = {
    'System prompt':    system_prompt,
    'Few-shot examples': few_shot,
    'Retrieved chunks': retrieved_chunks,
    'User message':     user_message,
}
total_input = 0
print(f'{"Component":<22} {"Tokens":>7}')
print('-'*31)
for name, text in INPUT_PARTS.items():
    n = count_tokens(text); total_input += n
    print(f'{name:<22} {n:>7}')
print(f'{"TOTAL INPUT":<22} {total_input:>7}')
output_tokens = count_tokens(typical_output)
print(f'{"OUTPUT":<22} {output_tokens:>7}')
print()
print(f'{"Model":<22} {"$/call":>10} {"$/1k calls":>12} {"$/day@10k":>12}')
print('-'*60)
for model, pr in MODEL_PRICING.items():
    c = cost_per_call(total_input, output_tokens, model)
    print(f'{model:<22} {c:>10.5f} {c*1000:>12.3f} {c*10000:>12.2f}')

## 2 · Conversation History — Biggest Cost Leak

Passing full conversation history on every call creates **linearly growing** costs per session.

In [ ]:
BASE_TOKENS   = count_tokens(system_prompt + user_message)
MODEL_        = 'gpt-4o-mini'
OUTPUT_TOK    = 50
TURNS = [
    ('What toppings are on Pepperoni?', 'Pepperoni, mozzarella, tomato sauce.'),
    ('Is there a gluten-free option?',  'Yes, GF base is +1.50.'),
    ('What is the minimum order?',      'Minimum order is 15.'),
    ('Do you deliver north zone Mon?',  'North zone Mon-Fri only.'),
    ('What time Sunday close?',         'Sunday 9pm.'),
] * 2

history = ''
session_cost = 0.0
print(f'{"Turn":>5} {"Input tokens":>14} {"Cost/call":>12} {"Session cost":>13}')
print('-'*50)
for i, (q, a) in enumerate(TURNS[:8], 1):
    input_tokens  = BASE_TOKENS + count_tokens(history)
    c             = cost_per_call(input_tokens, OUTPUT_TOK, MODEL_)
    session_cost += c
    print(f'{i:>5} {input_tokens:>14} {c:>12.6f} {session_cost:>13.6f}')
    history += f'User: {q}\nBot: {a}\n'
print(f'\nSession cost 8 turns (mid): ${session_cost:.4f}')
print(f'At 1,000 sessions/day: ${session_cost*1000:.2f}/day')

## 3 · Prefix Caching Savings

Static prefix (system prompt + few-shot) = 90% discounted when it remains identical across calls.
Put static content **first**, keep it **identical** across calls.

In [ ]:
STATIC   = system_prompt + '\n\n' + few_shot
DYNAMIC  = retrieved_chunks + '\n' + user_message
stat_tok = count_tokens(STATIC)
dyn_tok  = count_tokens(DYNAMIC)
tot_tok  = stat_tok + dyn_tok
CALLS    = 10_000
DISC     = 0.90
outp     = count_tokens(typical_output)
cost_nc  = cost_per_call(tot_tok, outp, 'gpt-4o-mini') * CALLS
eff_inp  = dyn_tok + stat_tok * (1 - DISC)
cost_c   = cost_per_call(eff_inp, outp, 'gpt-4o-mini') * CALLS
savings  = cost_nc - cost_c
print(f'Static prefix tokens : {stat_tok:>6}  ({stat_tok/tot_tok:.0%} of total)')
print(f'Dynamic tokens/call  : {dyn_tok:>6}')
print(f'Daily cost no cache  : ${cost_nc:>8.2f}')
print(f'Daily cost cached    : ${cost_c:>8.2f}')
print(f'Daily savings        : ${savings:>8.2f}  ({savings/cost_nc:.1%})')
print(f'Annual savings       : ${savings*365:>8.2f}')

## 4 · Latency Decomposition

```
Total latency = network_RTT + TTFT + (output_tokens x ms/token) + [optional post-processing]
```
TTFT scales with input token count. ms/token scales with model size.

In [ ]:
P = {'rtt': 50, 'ttft_per_1k': 120, 'ms_per_tok': 18, 'nli_ms': 300, 'sc_n': 5}

def est_lat(inp, outp, nli=False, sc=False):
    ttft = (inp/1000) * P['ttft_per_1k']
    gen  = outp * P['ms_per_tok']
    n    = P['sc_n'] if sc else 1
    tot  = P['rtt'] + (ttft + gen) * n + (P['nli_ms'] if nli else 0)
    return round(ttft), round(gen), round(tot/1000, 2)

configs = [
    ('Simple chat (500 input tokens)',     500,  80,  False, False),
    ('RAG pipeline (2k input)',           2000, 150, False, False),
    ('RAG + NLI verification',           2000, 150, True,  False),
    ('RAG + Self-consistency (5x)',       2000, 150, False, True),
    ('Full safety stack',                2000, 150, True,  True),
]
print(f'{"Configuration":<42} {"TTFT":>8} {"Gen":>8} {"Total":>8}')
print('-'*70)
for name, inp, out, nli, sc in configs:
    ttft, gen, tot = est_lat(inp, out, nli, sc)
    print(f'{name:<42} {ttft:>7}ms {gen:>7}ms {tot:>7}s')
print('\nSelf-consistency is the biggest latency multiplier — reserve for high-stakes queries.')

## Summary

| Lever | Impact | Action |
|---|---|---|
| Model tier | 10-100x cost | Use cheapest passing eval |
| Context length | Biggest driver | Summarise history; short chunks |
| Prefix caching | 40-80% reduction | Static content first, identical across calls |
| Self-consistency | 5x latency | Reserve for high-stakes only |
| Streaming | Perceived latency | Always stream user-visible text |

**Next:** [EvaluatingAISystems/notebook.ipynb](../EvaluatingAISystems/notebook.ipynb)